[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# sa_column and __table_args__ &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the naming convention, the classes, `hero_engine` and the
database. Run it first. `Gadget` grows through the tasks, and each time it does the old table is
dropped and taken out of the metadata before the class is written again, so run them in order. The
last cell removes the scratch folder.


In [1]:
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import CheckConstraint, Column, Integer, JSON, String, UniqueConstraint, event, insert, text
from sqlalchemy.dialects import postgresql, sqlite
from sqlalchemy.exc import IntegrityError
from sqlalchemy.schema import CreateTable
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


def table_sql(model):
    """The CREATE TABLE a table model describes, written for SQLite with no database anywhere."""
    return str(CreateTable(model.__table__).compile(dialect=sqlite.dialect())).strip()

SQLModel.metadata.naming_convention = {                             # every constraint gets a name from its shape
    "ix": "ix_%(table_name)s_%(column_0_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "pk": "pk_%(table_name)s",
}


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes |",
          "naming convention:", len(SQLModel.metadata.naming_convention), "rules")


sqlmodel 0.0.42 | 8 heroes | naming convention: 5 rules


**1.** A gadget with a dictionary in it.


In [2]:
class Gadget(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=60)
    settings: dict = Field(default_factory=dict, sa_column=Column(JSON))


SQLModel.metadata.create_all(engine)
print(table_sql(Gadget))


CREATE TABLE gadget (
	id INTEGER NOT NULL, 
	name VARCHAR(60) NOT NULL, 
	settings JSON, 
	CONSTRAINT pk_gadget PRIMARY KEY (id)
)


`dict` has no column type of its own, so the column is written by hand as `Column(JSON)`.


**2.** Settings written and read back.


In [3]:
with Session(engine) as session:
    session.add(Gadget(name="Grapple line", settings={"mode": "stealth", "charges": 2}))
    session.commit()

with Session(engine) as session:
    grapple = session.exec(select(Gadget)).one()
    print(grapple.settings, type(grapple.settings).__name__, "| one value:", grapple.settings["mode"])


{'mode': 'stealth', 'charges': 2} dict | one value: stealth


A dictionary in and a dictionary out. SQLAlchemy wrote it as JSON text and read it back as Python.


**3.** A serial, unique and indexed, written inside the column.


In [4]:
Gadget.__table__.drop(engine)                                       # the old shape goes first
SQLModel.metadata.remove(Gadget.__table__)


class Gadget(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=60)
    serial: str = Field(sa_column=Column(String(20), unique=True, index=True, nullable=False))
    settings: dict = Field(default_factory=dict, sa_column=Column(JSON))


SQLModel.metadata.create_all(engine)
print(table_sql(Gadget))
print("indexes:", sorted(index.name for index in Gadget.__table__.indexes))


CREATE TABLE gadget (
	id INTEGER NOT NULL, 
	name VARCHAR(60) NOT NULL, 
	serial VARCHAR(20) NOT NULL, 
	settings JSON, 
	CONSTRAINT pk_gadget PRIMARY KEY (id)
)
indexes: ['ix_gadget_serial']


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sqlmodel/main.py:722: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.Gadget, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)


`unique`, `index` and `nullable` all went inside the `Column`, since passing any of them to `Field`
beside `sa_column` raises. The index is named by the convention in Setup.


**4.** A check the database enforces.


In [5]:
Gadget.__table__.drop(engine)
SQLModel.metadata.remove(Gadget.__table__)


class Gadget(SQLModel, table=True):
    __table_args__ = (CheckConstraint("charges >= 0", name="charges_are_not_negative"),)

    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=60)
    serial: str = Field(sa_column=Column(String(20), unique=True, index=True, nullable=False))
    charges: int = Field(default=0)
    settings: dict = Field(default_factory=dict, sa_column=Column(JSON))


SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    session.add(Gadget(name="Smoke pellet", serial="SP-1", charges=-2))
    try:
        session.commit()
    except IntegrityError as error:
        print("refused :", str(error).splitlines()[0])
    session.rollback()

    session.add(Gadget(name="Smoke pellet", serial="SP-1", charges=3))
    session.commit()
    print("accepted:", session.exec(select(Gadget)).one().charges, "charges")


refused : (sqlite3.IntegrityError) CHECK constraint failed: ck_gadget_charges_are_not_negative
accepted: 3 charges


The refusal names the constraint, because the constraint was given a name.


**5.** A column the table fills in.


In [6]:
Gadget.__table__.drop(engine)
SQLModel.metadata.remove(Gadget.__table__)


class Gadget(SQLModel, table=True):
    __table_args__ = (CheckConstraint("charges >= 0", name="charges_are_not_negative"),)

    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=60)
    serial: str = Field(sa_column=Column(String(20), unique=True, index=True, nullable=False))
    charges: int = Field(default=0)
    checked_by: str | None = Field(default=None,
                                   sa_column=Column(String(40), server_default=text("'the quartermaster'")))
    settings: dict = Field(default_factory=dict, sa_column=Column(JSON))


SQLModel.metadata.create_all(engine)
with engine.begin() as connection:
    connection.execute(text("INSERT INTO gadget (name, serial, charges, settings) "
                            "VALUES ('Smoke pellet', 'SP-2', 3, '{}')"))

with Session(engine) as session:
    print("checked_by:", session.exec(select(Gadget)).one().checked_by)


checked_by: the quartermaster


The row was written by plain SQL, which knows nothing about the model, and the table filled the
column in.


**6.** What every rule is called.


In [7]:
print("constraints:", sorted(rule.name for rule in Gadget.__table__.constraints if rule.name))
print("indexes    :", sorted(index.name for index in Gadget.__table__.indexes))
# pk_gadget and ix_gadget_serial came from the convention in Setup; uq_gadget_serial too, from the
# unique=True inside the Column. charges_are_not_negative is the one that was named by hand.


constraints: ['ck_gadget_charges_are_not_negative', 'pk_gadget']
indexes    : ['ix_gadget_serial']


Every rule has a name, and only one of them was typed. That is what makes the **Migrations**
notebook's work possible on SQLite, where changing a constraint means rebuilding the table.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [sa_column and __table_args__](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/11-sa-column-and-table-args.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
